# Load test: 50 concurrent users against the served API

Runs on **Kaggle, GPU T4 x2, Internet ON**, with the `civil-code-corpus` dataset added.

The setup under test is the one chosen for serving: the **FastAPI app** (PII redaction, retrieval, prompt, metrics) calling **Qwen3-8B-AWQ on vLLM**. Locust drives it twice:

1. **1 user for 2 min**: the latency baseline, with no contention
2. **50 users for 5 min**: the rubric's load level

Output: `reports/locust_u1.html`, `reports/locust_u50.html`, the CSVs, `reports/load_test.md` (summary table) and `reports/load_metrics.txt` (the API's own Prometheus metrics after the run).

Long cells write to log files; if the display freezes, reload the page (F5). The work continues. Budget: about 45 GPU-minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and corpus

In [ ]:
!git clone -q https://github.com/ZeinabMahfouz/The-Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1
!mkdir -p data/interim eval_out && cp "$(find /kaggle/input -name civil_code.json | head -1)" data/interim/
# The two md5 values must match
!md5sum data/interim/civil_code.json
!grep -A3 "path: data/interim/civil_code.json" dvc.lock | grep "md5:"

## 2. vLLM in its own environment

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
!/kaggle/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import os, signal, subprocess, time, urllib.request

def start(cmd, log_path, env=None):
    return subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def wait_ready(url, log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen(url, timeout=5); print(f"ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- log:"); os.system(f"tail -n 30 {log_path}"); return False

## 3. API environment (FastAPI app + Locust)

In [ ]:
!pip install -q fastapi uvicorn "qdrant-client==1.19.1" sentence-transformers pyyaml "openai==1.109.1" \
    langfuse==4.15.6 prometheus-client==0.26.0 locust
!pip install -q -e . --no-deps

## 4. Build the index (same chunking and embedding as `dvc repro`, on GPU)

In [ ]:
!python scripts/chunk_corpus.py --corpus data/interim/civil_code.json --out data/interim/chunks.json --params params.yaml
!python scripts/embed_and_index.py --chunks data/interim/chunks.json --params params.yaml --docs-dir data/documents 2>&1 | grep -v "Batches:" | tail -n 4

## 5. Start vLLM (AWQ) and the API

In [ ]:
# 0.6 GPU memory: AWQ weights are only ~2.9 GiB per GPU; the rest is KV cache for concurrent requests,
# with room left for the API's BGE-M3 on cuda:0.
vllm = start(["/kaggle/vllm-env/bin/vllm", "serve", "Qwen/Qwen3-8B-AWQ",
              "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "8192",
              "--gpu-memory-utilization", "0.6", "--port", "8000"],
             "/kaggle/working/vllm.log", env={"PYTHONPATH": ""})
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm.log")

In [ ]:
# The same app as the Docker image; VLLM_BASE_URL switches generation to the vLLM server.
api = start(["uvicorn", "egyptian_civil_code_rag.api:app", "--host", "0.0.0.0", "--port", "8001"],
            "/kaggle/working/api.log",
            env={"VLLM_BASE_URL": "http://localhost:8000/v1", "GEN_MODEL": "Qwen/Qwen3-8B-AWQ"})
wait_ready("http://localhost:8001/health", "/kaggle/working/api.log", minutes=10)
!curl -s http://localhost:8001/health; echo
!curl -s -X POST http://localhost:8001/ask -H "Content-Type: application/json" -d '{"question": "Is Article 400 still in force?"}'; echo
!grep "\[startup\]" /kaggle/working/api.log

## 6. Locust: 1 user (baseline), then 50 users

In [ ]:
!locust -f load_test/locustfile.py --host http://localhost:8001 --headless -u 1 -r 1 -t 2m \
    --csv reports/locust_u1 --html reports/locust_u1.html --only-summary > eval_out/locust_u1.log 2>&1; tail -n 8 eval_out/locust_u1.log

In [ ]:
# ~6 min: 5 users/s ramp to 50, then 5 minutes at 50.
!locust -f load_test/locustfile.py --host http://localhost:8001 --headless -u 50 -r 5 -t 5m \
    --csv reports/locust_u50 --html reports/locust_u50.html --only-summary > eval_out/locust_u50.log 2>&1; tail -n 8 eval_out/locust_u50.log

In [ ]:
!python scripts/locust_summary.py reports/locust_u1 reports/locust_u50 --out reports/load_test.md
# The API's own view of the same load (latency histogram per stage, request counts, tokens)
!curl -s http://localhost:8001/metrics | grep "^rag_" > reports/load_metrics.txt; wc -l reports/load_metrics.txt

## 7. Download

In [ ]:
!zip -qr /kaggle/working/load_test_results.zip reports/locust_u1* reports/locust_u50* reports/load_test.md \
    reports/load_metrics.txt eval_out/locust_*.log
!cp /kaggle/working/api.log /kaggle/working/vllm.log eval_out/ && zip -qr /kaggle/working/load_test_results.zip eval_out/api.log eval_out/vllm.log
%cd /kaggle/working
from IPython.display import FileLink
display(FileLink("load_test_results.zip"))